In [3]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns

import sklearn
sklearn.set_config(display='text')
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier # 앙상블 스태킹 알고리즘을 사용하기 위해 import 한다.

스태킹(stacking)

스태킹은 베이스 학습기와 메타 학습기로 구성되어 있고 베이스 학습기와 메타 학습긴 서포트 벡터 머신, 랜덤 포레스트 같은 학습 모델이다. 베이스 학습기가 먼저 학습한 후 메타 학습기는 베이스 학습기의 예측 피쳐 데이터로 활용해서 최종 예측을 한다.

와인 데이터를 사용해서 와인 종류를 분류하기 위해 데이터를 불러오고 표준화 한다.

In [2]:
# 데이터 불러오기
raw_data = datasets.load_wine() # 사이킷런 라이브러리가 제공하는 와인 데이터를 불러온다.
# print(raw_data)

# 피쳐, 레이블 데이터 저장
xData = raw_data.data # 피쳐 데이터를 저장한다.
yData = raw_data.target # 피쳐 데이터에 따른 레이블을 저장한다.
# print(xData.shape, yData.shape)

# 학습 데이터와 테스트 데이터로 분할
x_train, x_test, y_train, y_test = train_test_split(xData, yData, random_state=0)
# print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

# 데이터 표준화(정규화)
scaler = StandardScaler() # 표준화 스케일러 객체를 만든다.
x_train = scaler.fit_transform(x_train) # 학습 데이터를 표준화 스케일러로 표준화하고 적용한다.
x_test = scaler.transform(x_test) # 테스트 데이터를 학습 데이터로 표준화한 스케일러에 적용한다.

모델을 생성하고 학습시킨다.

In [7]:
# 베이스 학습기를 만든다.
model_sv = SVC(kernel='rbf', C=0.1, probability=True) # 앙상블 스태킹 알고리즘에서 사용할 베이스 학습기로 k-최근접 이웃 개별 분류기를 만든다.
model_nb = GaussianNB() # 앙상블 스태킹 알고리즘에서 사용할 베이스 분류기로 가우시안 나이브 베이즈 개별 학습기를 만든다.

# 메타 학습기를 만든다.
model_lr = LogisticRegression(l1_ratio=0.0, C=0.1, solver='saga') # 앙상블 스태킹 알고리즘에서 사용할 메타 학습기로 k-최근접 이웃 개별 분류기를 만든다.

# 베이스 학습기와 메타 학습기를 사용해서 앙상블 스태킹 모델을 만든다.
# estimators 속성값으로 베이스 학습기를 지정하고 final_estimator 속성값으로 메타 학습기를 지정한다.
model = StackingClassifier(estimators=[('nb', model_nb), ('sv', model_sv)], final_estimator=model_lr).fit(x_train, y_train)

학습된 모델로 테스트 데이터를 예측한다.

In [8]:
predict = model.predict(x_test) # predict() 메소드의 인수로 표준화된 테스트 데이터(x_test)를 넘겨서 서포트 벡터 머신 모델을 예측한다.
print(predict)

[0 2 1 0 1 1 0 2 1 1 2 2 0 1 2 1 0 0 2 0 0 0 0 1 1 1 1 1 1 2 0 0 1 0 0 0 2
 1 1 2 0 0 1 1 1]


In [9]:
predict_proba = model.predict_proba(x_test) # predict_proba() 메소드의 인수로 표준화된 테스트 데이터(x_test)를 넘겨서 각 클래스에 속할 확률로 예측한다.
print(predict_proba)

[[0.82503651 0.09458044 0.08038305]
 [0.09488455 0.1263206  0.77879485]
 [0.09912483 0.82683009 0.07404508]
 [0.82403168 0.09517433 0.08079399]
 [0.09729338 0.82389532 0.0788113 ]
 [0.10713877 0.80684529 0.08601593]
 [0.82432669 0.09489682 0.08077649]
 [0.09323169 0.1240655  0.7827028 ]
 [0.08589103 0.84504302 0.06906595]
 [0.08617202 0.84387747 0.06995051]
 [0.09677598 0.13086272 0.77236129]
 [0.0977436  0.13155598 0.77070042]
 [0.82575975 0.09413011 0.08011015]
 [0.35205034 0.52756132 0.12038834]
 [0.09350284 0.12396873 0.78252843]
 [0.08573257 0.84535425 0.06891317]
 [0.80514226 0.10710125 0.08775649]
 [0.82038688 0.09687372 0.0827394 ]
 [0.14120503 0.37138809 0.48740688]
 [0.82542369 0.0943963  0.08018001]
 [0.45196008 0.42032812 0.1277118 ]
 [0.8103098  0.10492313 0.08476707]
 [0.76022224 0.14196983 0.09780793]
 [0.08735216 0.8430778  0.06957003]
 [0.09406978 0.82374903 0.08218119]
 [0.08619107 0.84450185 0.06930709]
 [0.08827356 0.84122285 0.07050359]
 [0.08562523 0.84536278 0.06

학습된 모델을 평가한다.

In [10]:
# 혼동 행렬
# confusion_matrix() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 혼동 행렬을 출력한다.
confusion = confusion_matrix(y_test, predict)
print(confusion)

[[16  0  0]
 [ 1 19  1]
 [ 0  0  8]]


In [11]:
# 분류 리포트
# classification_report() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 분류 리포트를 출력한다.
classification = classification_report(y_test, predict, target_names=raw_data.target_names)
print(classification)

              precision    recall  f1-score   support

     class_0       0.94      1.00      0.97        16
     class_1       1.00      0.90      0.95        21
     class_2       0.89      1.00      0.94         8

    accuracy                           0.96        45
   macro avg       0.94      0.97      0.95        45
weighted avg       0.96      0.96      0.96        45



In [12]:
# 정확도 평가
# accuracy_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 정확도를 계산한다.
accuracy = accuracy_score(y_test, predict)
print(accuracy)

0.9555555555555556


In [13]:
# 정밀도 평가
# precision_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 정밀도를 계산한다.
precision = precision_score(y_test, predict, average=None)
print(precision)

[0.94117647 1.         0.88888889]


In [14]:
# 재현율 평가
# recall_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 재현율을 계산한다.
recall = recall_score(y_test, predict, average=None)
print(recall)

[1.        0.9047619 1.       ]


In [15]:
# f1 score 평가
# f1_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 f1 score를 계산한다.
f1 = f1_score(y_test, predict, average=None)
print(f1)

[0.96969697 0.95       0.94117647]
